# Метрики и визуализация

**II Саммит талантов · трек ИИ · Д2П2 · Иван Эйдлин**

Что смотрим:

1. accuracy / precision / recall / F1 на живом примере
2. Confusion matrix — картинкой
3. Scatter классов по двум признакам
4. Histogram распределения признака
5. MSE / MAE / R² для регрессии
6. Переобучение — как оно выглядит глазами

## 1. Данные

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
df = iris.frame
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
df.head()

## 2. Метрики классификации

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'accuracy  = {accuracy_score(y_test, y_pred):.3f}')
print(f'precision = {precision_score(y_test, y_pred, average="macro"):.3f}')
print(f'recall    = {recall_score(y_test, y_pred, average="macro"):.3f}')
print(f'f1        = {f1_score(y_test, y_pred, average="macro"):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Confusion matrix — картинкой

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(ax=ax, colorbar=False)
ax.set_title('Confusion matrix · LogReg')
plt.show()

## 3. Scatter классов по двум признакам

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for cls in sorted(df['target'].unique()):
    sub = df[df['target'] == cls]
    ax.scatter(sub['petal length (cm)'], sub['petal width (cm)'],
               label=iris.target_names[cls], alpha=0.7)
ax.set_xlabel('petal length')
ax.set_ylabel('petal width')
ax.set_title('Ирисы в пространстве двух признаков')
ax.legend()
plt.show()

## 4. Histogram признака

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(df['sepal length (cm)'], bins=20, edgecolor='black')
ax.set_xlabel('sepal length')
ax.set_ylabel('кол-во цветков')
ax.set_title('Распределение длины чашелистика')
plt.show()

## 5. Метрики регрессии

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

cali = fetch_california_housing(as_frame=True)
Xr, yr = cali.data, cali.target
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.3, random_state=42)

reg = LinearRegression().fit(Xr_tr, yr_tr)
yr_pred = reg.predict(Xr_te)

print(f'MSE = {mean_squared_error(yr_te, yr_pred):.3f}')
print(f'MAE = {mean_absolute_error(yr_te, yr_pred):.3f}')
print(f'R²  = {r2_score(yr_te, yr_pred):.3f}')

## 6. Переобучение глазами

Смотрим, как растёт качество на train и падает на test при увеличении глубины решающего дерева.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = range(1, 20)
train_acc, test_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(m.score(X_train, y_train))
    test_acc.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(depths, train_acc, 'o-', label='train')
ax.plot(depths, test_acc,  's-', label='test')
ax.set_xlabel('max_depth')
ax.set_ylabel('accuracy')
ax.set_title('Переобучение решающего дерева')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

> Ключевой момент: train-кривая растёт монотонно, а test — останавливается и может падать. Это и есть переобучение на глаз.

## 7. Что сказать классу

1. Одной метрики не хватает. Accuracy врёт на несбалансированных классах.
2. Precision и recall — это два взгляда на одну модель: «не ошибаюсь ли я, когда говорю да» vs «не пропускаю ли я важное».
3. F1 — это компромисс между ними. Когда важны оба — смотрим F1.
4. В регрессии MAE честнее для интерпретации, MSE — жёстче штрафует большие промахи, R² — сравнивает с «просто среднее».
5. Если train ≫ test — модель выучила данные, а не зависимость. Лечится ограничением сложности.